### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** You don't just calculate eigenvalues; you see the **Condition Number** which dictates if your model will even converge. You don't just use MSE; you know it's a specific MLE derived from the assumption of Gaussian noise. You treat probabilities as **beliefs** to be updated (Bayesian), not just long-run frequencies.

**Mental Model:** 
- **SVD:** A prism that splits light into its core component colors (singular values).
- **Hessian:** A 3D topographical map that tells you how steep the valley is in every possible direction.
- **Bayes' Theorem:** A 'belief-updating machine' that combines what you knew (Prior) with what you saw (Likelihood).

**Common Junior Pitfall:** Running PCA on uncentered or unstandardized data. PCA finds the direction of maximum *scale*, not maximum *information*. Without scaling, the feature with the largest numbers always wins, even if it's junk noise.

---


## 1. Linear Algebra: The Geometric Reality

In ML, a matrix is a **Linear Transformation**. It rotates, stretches, and skews vectors. 

### Principal Components from Scratch
Eigenvectors point in the direction of maximum variance (the 'interesting' directions), and eigenvalues tell you how much information (variance) each direction carries.


## 📑 Table of Contents

  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1. Linear Algebra: The Geometric Reality](#1-linear-algebra-the-geometric-reality)
  * [Principal Components from Scratch](#principal-components-from-scratch)
* [1.1 Singular Value Decomposition (SVD)](#11-singular-value-decomposition-svd)
* [2. Calculus & Optimization](#2-calculus-optimization)
* [2.1 The Hessian Matrix & Second-Order Optimization](#21-the-hessian-matrix-second-order-optimization)
  * [Newton's Method](#newtons-method)
* [3. Probability & Statistics (MLE)](#3-probability-statistics-mle)
* [3.1 Bayesian Probability — Prior, Likelihood, Posterior](#31-bayesian-probability-prior-likelihood-posterior)
  * [The Link to Regularization](#the-link-to-regularization)
* [4. Data Reality: Imputation](#4-data-reality-imputation)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: Why use SVD instead of Eigendecomposition?](#q1-why-use-svd-instead-of-eigendecomposition)
  * [Q2: Why is Newton's Method better but also worse than SGD?](#q2-why-is-newtons-method-better-but-also-worse-than-sgd)
* [🔨 Practical Practice](#practical-practice)
  * [Tier 1: Beginner](#tier-1-beginner)
  * [Tier 2: Intermediate](#tier-2-intermediate)
  * [Tier 3: Advanced](#tier-3-advanced)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
X = np.random.multivariate_normal(mean=[0, 0], cov=[[5, 4], [4, 5]], size=500)
X_centered = X - np.mean(X, axis=0)
cov_matrix = np.cov(X_centered, rowvar=False)
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

print(f"PC1 explains {eigenvalues[0]/sum(eigenvalues):.1%} of the variance.")

"""
What just happened?
We derived the 'bones' of the dataset. Eigenvectors are the principal components. 
The first component captures over 80% of the movement, meaning we could safely 
ignore the second dimension with minimal loss of information.
"""

## 1.1 Singular Value Decomposition (SVD)

Eigendecomposition requires a square matrix. **SVD** is universal. Every $m \times n$ matrix $A$ can be decomposed into:
$$A = U \Sigma V^T$$
1. **$U$:** Left singular vectors (Rotation in output space).
2. **$\Sigma$:** Singular values (Scaling).
3. **$V^T$:** Right singular vectors (Rotation in input space).

**Why this matters for PCA:**
PCA is physically equivalent to performing SVD on your centered data matrix $X$. The singular values are the square roots of the eigenvalues.

**Condition Number:** The ratio $\sigma_{max} / \sigma_{min}$. A high condition number means the matrix is 'ill-conditioned' — small errors in your data lead to massive errors in your model's weights. This is why standardization is numerically mandatory.

📈 **Production Signal:** SVD is used for **Image Compression** and **Recommender Engines** (like Netflix's original algorithm), where we factorize a huge User-Movie matrix into low-rank representations.


In [ ]:
from scipy.datasets import face
img = face(gray=True).astype(float)

U, S, Vt = np.linalg.svd(img, full_matrices=False)

def compress_image(k):
    reconst = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    return reconst

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, k in enumerate([5, 20, 100, 512]):
    axes[i].imshow(compress_image(k), cmap='gray')
    axes[i].set_title(f"k={k} Singular Values")
    axes[i].axis('off')
plt.show()

"""
What just happened?
We compressed an image using Truncated SVD. By keeping only the top-k singular values, 
we keep the 'global' structure (k=5) and gradually add high-frequency detail. 
In ML, we use this to throw away noise while keeping signal.
"""

## 2. Calculus & Optimization

AutoDiff is the engine of PyTorch. It applies the **Chain Rule** systematically by tracking operations on a computational graph.


In [ ]:
class Value:
    def __init__(self, data, _children=()):
        self.data, self.grad, self._prev = data, 0.0, set(_children)
        self._backward = lambda: None
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward; return out
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward; return out
    def backward(self):
        topo, visited = [], set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev: build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo): node._backward()

x = Value(3.0); y = Value(-4.0)
f = x * y + 2.0; f.backward()
print(f"df/dx: {x.grad} (y), df/dy: {y.grad} (x)")

"""
What just happened?
We built a recursive AutoDiff engine. This 'Value' class knows how to distribute 
the gradients from the output back to its parents using the chain rule.
"""

## 2.1 The Hessian Matrix & Second-Order Optimization

Gradient Descent only knows the **slope** (1st derivative). The **Hessian Matrix** $H = \nabla^2 L$ knows the **curvature** (2nd derivative).

**Key Insight:** 
If the Hessian is 'badly conditioned' (Ratio $\lambda_{max}/\lambda_{min}$ is high), the loss landscape is a narrow, squashed valley. SGD will oscillate violently between the walls, forcing you to use a tiny learning rate.

### Newton's Method
Instead of taking a step proportional to the gradient, we take a step scaled by the **inverse of the Hessian**:
$$W_{t+1} = W_t - H^{-1} \nabla L$$

**The 7B Parameter Problem:** 
For an LLM with 7 billion parameters, the Hessian is $7B \times 7B$. Storing it would require ~196 Exabytes of VRAM. This is why we use **Adam**, which approximates the diagonal of the Hessian instead of the whole thing.


In [ ]:
def f(x, y): return x**2 + 50*y**2 # A steep valley
def grad_f(x, y): return np.array([2*x, 100*y])
def hessian_f(x, y): return np.array([[2, 0], [0, 100]])

w = np.array([10.0, 1.0])

# Newton Step
H = hessian_f(*w)
g = grad_f(*w)
step = np.linalg.inv(H) @ g
w_newton = w - step

print(f"Original Point: {w}")
print(f"Newton Step Result: {w_newton} (Found minimum in exactly 1 step!)")

"""
What just happened?
Newton's Method used the curvature info to realize that the 'y' direction is much 
steeper and needs a smaller relative step than 'x'. By dividing by the Hessian, 
it reached the global minimum of this quadratic surface in a single leap.
"""

## 3. Probability & Statistics (MLE)

Loss functions (MSE, Cross-Entropy) are derived directly from **Maximum Likelihood Estimation (MLE)**. 
- MSE is MLE assuming **Gaussian Noise**.
- Cross-Entropy is MLE assuming a **Bernoulli Distribution**.


## 3.1 Bayesian Probability — Prior, Likelihood, Posterior

Bayesian ML treats weights as random variables, not single fixed numbers. 
$$P(\theta | D) = \frac{P(D | \theta) \cdot P(\theta)}{P(D)}$$
- **Likelihood $P(D|\theta)$:** What the data tells us.
- **Prior $P(\theta)$:** What we believed before seeing data.
- **Posterior $P(\theta|D)$:** Our updated belief.

### The Link to Regularization
When we assume a **Gaussian Prior** on our weights (believing they should be small and near zero), and we find the Maximum A Posteriori (MAP) estimate, the resulting formula is mathematically identical to **L2 Weight Decay (Regularization)**.

📈 **Production Signal:** Bayesian estimation is mandatory for **Thompson Sampling** in Recommendation Systems (deciding whether to show a new ad or a proven one).


In [ ]:
from scipy.stats import beta

# Prior belief: 50/50 chance (Beta(2, 2))
a, b = 2, 2
x_axis = np.linspace(0, 1, 100)

# Observation: We see 8 heads and 2 tails (k=8, n=10)
k, n = 8, 10

plt.plot(x_axis, beta.pdf(x_axis, a, b), '--', label='Prior: Beta(2,2)')
plt.plot(x_axis, beta.pdf(x_axis, a+k, b+n-k), label='Posterior: Beta(10,4)')
plt.title("Bayesian Updating: My Belief Shifted Right after Seeing 8 Heads")
plt.legend(); plt.show()

"""
What just happened?
We implemented a 'conjugate prior'. Notice how simple the math is: we just added 
the counts to the prior parameters. The distribution 'sharpened' around the new 
empirical evidence (0.8).
"""

## 4. Data Reality: Imputation

In production, 30% of your values might be missing. How you fill them dictates your model's bias.


In [ ]:
from sklearn.metrics.pairwise import nan_euclidean_distances

def knn_impute(X, k=2):
    X_imp = X.copy(); dists = nan_euclidean_distances(X, X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            if np.isnan(X[i, j]):
                neighbors = X[np.argsort(dists[i])[1:], j]
                X_imp[i, j] = np.nanmean(neighbors[:k])
    return X_imp

"""
What just happened?
We implemented a KNN Imputer. It looks at the most similar rows (based on other 
available features) to estimate the missing value. This preserves correlation 
much better than simple mean imputation.
"""

---
## ✅ Knowledge Check

### Q1: Why use SVD instead of Eigendecomposition?
<details><summary>👉 Answer</summary>
Eigendecomposition only works for square matrices. SVD works for any shape matrix ($m \times n$), which is essential because data matrices are almost always rectangular (Users vs Items, Features vs Samples).
</details>

### Q2: Why is Newton's Method better but also worse than SGD?
<details><summary>👉 Answer</summary>
Newton's Method converges in fewer steps because it uses second-order curvature information to aim directly for the 'floor' of the valley. However, calculating and inverting the Hessian matrix costs $O(N^3)$ where $N$ is the number of parameters — physically impossible for deep neural networks.
</details>


---
## 🔨 Practical Practice

### Tier 1: Beginner
1. Calculate the **Singular Value Decomposition** of a $2 	imes 3$ matrix and manually reconstruct it using all singular values. Prove they are identical.
2. Implement `Forward Difference` numeric differentiation ($[f(x+h) - f(x)]/h$) and compare the gradient to our `AutoDiff` engine for $f(x) = x^2$.

### Tier 2: Intermediate
1. **SVD Image Reconstruction:** Plot a curve of 'Mean Squared Error' between the original image and its SVD reconstruction for $k \in [1, 5, 10, 50, 100, 500]$. At what $k$ is the image visually indistinguishable from the original?
2. **The Bayesian Shifter:** Observe 100 coin flips where 70 are heads. Start with a very biased 'Anti-Heads' prior (Beta(10, 2)). How many flips does it take before the Posterior mean finally crosses 0.5?

### Tier 3: Advanced
1. **Gradient descent vs Newton:** Implement a 2D Rosenbrock function (the 'banana' function). Perform 100 steps of SGD ($lr=0.001$) and 5 steps of Newton. Plot the paths. Why does SGD struggle on this surface while Newton thrives?
2. **MAP from Scratch:** Derive the loss function for a Linear Regression model assuming a Laplacian Prior ($P(w) = e^{-|w|}$) instead of a Gaussian. What common regularization technique did you just 'discover' through Bayesian logic?
